In [ ]:
import importlib
import sys

# Force reload of helpers module to pick up latest changes
if 'helpers' in sys.modules:
    importlib.reload(sys.modules['helpers'])
    importlib.reload(sys.modules['helpers.database'])
    importlib.reload(sys.modules['helpers.logging_config'])


In [0]:
# ==============================================================================
# CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================

from helpers import (
    create_connection, load_bronze_table, write_gold_table, setup_logger,
    transform_service_fact, transform_rental_fact
)
import time

# Setup logging
logger = setup_logger("load_facts")

# Initialize connection
c = create_connection(spark, dbutils)
logger.info("=" * 70)
logger.info("FACT LOAD JOB STARTED")
logger.info("=" * 70)


In [0]:
# ==============================================================================
# BRONZE LAYER: Load source tables (in-memory DataFrames)
# ==============================================================================

service_bronze = load_bronze_table(c, "service")
rental_bronze = load_bronze_table(c, "rental")
staff_bronze = load_bronze_table(c, "staff")
inventory_bronze = load_bronze_table(c, "inventory")
payment_bronze = load_bronze_table(c, "payment")


In [0]:
# ==============================================================================
# GOLD: FACT_SERVICE
# ==============================================================================

logger.info("GOLD: Building fact_service")
start_time = time.time()

fact_service = transform_service_fact(service_bronze)

write_gold_table(fact_service, "fact_service", mode="overwrite", partition_by=["service_date"])
logger.info(f"GOLD: fact_service completed in {time.time() - start_time:.2f}s")

In [ ]:
# ==============================================================================
# GOLD: FACT_RENTAL
# ==============================================================================

logger.info("GOLD: Building fact_rental")
start_time = time.time()

fact_rental = transform_rental_fact(
    rental_bronze, staff_bronze, inventory_bronze, payment_bronze
)

write_gold_table(fact_rental, "fact_rental", mode="overwrite", partition_by=["rental_date"])
logger.info(f"GOLD: fact_rental completed in {time.time() - start_time:.2f}s")

# ==============================================================================
# JOB COMPLETION
# ==============================================================================
logger.info("=" * 70)
logger.info("FACT LOAD JOB COMPLETED SUCCESSFULLY")
logger.info("=" * 70)